In [1]:
import pandas as pd
import numpy as np
import re
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:

users_pivot = None
recipe_matrix = None
popularity_df = None
recipes = None
recipe_map = None
recipe_reverse_map = None
user_map = None 

In [27]:
def load_data():
    global users_pivot, users_pivot_array, popularity_df, recipes, recipe_map, recipe_reverse_map, user_map

    print("Загрузка данных...")
    recipes_df = pd.read_csv("RAW_recipes.csv")
    interactions = pd.read_csv("RAW_interactions.csv")

    # Приведение типов к int64
    interactions = interactions.astype({'user_id': 'int64', 'recipe_id': 'int64'})
    recipes_df = recipes_df.astype({'id': 'int64'})

    # Фильтрация топ-пользователей и рецептов
    all_user_counts_original = interactions.groupby('user_id')['rating'].count()
    all_recipe_counts_original = interactions.groupby('recipe_id')['rating'].count()

    top_users_original = all_user_counts_original.nlargest(5000).index
    top_recipes_original = all_recipe_counts_original.nlargest(3000).index

    filtered_interactions_original = interactions[
        (interactions['user_id'].isin(top_users_original)) & 
        (interactions['recipe_id'].isin(top_recipes_original))
    ]

    first_30_recipes_from_filtered = filtered_interactions_original['recipe_id'].drop_duplicates().head(30)
    selected_recipes_info = recipes_df[recipes_df['id'].isin(first_30_recipes_from_filtered)]
    rated_recipe_names = selected_recipes_info.set_index('id').loc[first_30_recipes_from_filtered, 'name'].tolist()

    # Добавляем нового пользователя
    new_user_id = 666666
    new_user_recipe_ids = [int(x) for x in first_30_recipes_from_filtered.tolist()]
    print(f"ID рецептов для нового пользователя {new_user_id}: {new_user_recipe_ids}")

    new_user_data = []
    for recipe_id in new_user_recipe_ids:
        random_rating = np.random.choice([3.0, 4.0, 5.0])
        new_user_data.append({
            'user_id': new_user_id,
            'recipe_id': recipe_id,
            'rating': random_rating
        })

    new_user_df = pd.DataFrame(new_user_data)
    new_user_df = new_user_df.astype({'user_id': 'int64', 'recipe_id': 'int64', 'rating': 'float64'})

    interactions_with_new_user = pd.concat([interactions, new_user_df], ignore_index=True)
    interactions_with_new_user = interactions_with_new_user.astype({'user_id': 'int64', 'recipe_id': 'int64'})

    # Финальная фильтрация
    all_user_counts = interactions_with_new_user.groupby('user_id')['rating'].count()
    all_recipe_counts = interactions_with_new_user.groupby('recipe_id')['rating'].count()

    top_users = all_user_counts.nlargest(5000).index
    top_recipes = all_recipe_counts.nlargest(3000).index

    required_users = {new_user_id}
    required_recipes = set(new_user_recipe_ids)

    final_users = set(top_users).union(required_users)
    final_recipes = set(top_recipes).union(required_recipes)

    print(f"Количество финальных пользователей: {len(final_users)} (включая {new_user_id})")
    print(f"Количество финальных рецептов: {len(final_recipes)} (включая {new_user_recipe_ids[:3]}...)")

    filtered_interactions = interactions_with_new_user[
        (interactions_with_new_user['user_id'].isin(final_users)) & 
        (interactions_with_new_user['recipe_id'].isin(final_recipes))
    ]
    print(f"Количество строк в filtered_interactions: {len(filtered_interactions)}")

    # Агрегация
    user_recipe_agg = filtered_interactions.groupby(['user_id', 'recipe_id'])['rating'].mean().reset_index()
    user_recipe_agg = user_recipe_agg.astype({'user_id': 'int64', 'recipe_id': 'int64'})

    unique_users = user_recipe_agg['user_id'].unique()
    unique_recipes = user_recipe_agg['recipe_id'].unique()

    # Построение pivot по реальным ID, затем упорядочивание
    pivot_table = user_recipe_agg.pivot_table(
        index='user_id',
        columns='recipe_id',
        values='rating',
        fill_value=0
    )
    pivot_table = pivot_table.reindex(index=unique_users, columns=unique_recipes, fill_value=0)

    # Сброс индекса → теперь строки 0, 1, 2, ..., N-1
    users_pivot = pivot_table.reset_index(drop=True)
    users_pivot_array = users_pivot.values  # для cosine_similarity

    # Маппинги
    user_map = {user_id: idx for idx, user_id in enumerate(unique_users)}
    recipe_map = {recipe_id: idx for idx, recipe_id in enumerate(unique_recipes)}
    recipe_reverse_map = {idx: recipe_id for recipe_id, idx in recipe_map.items()}

    # Популярность
    recipe_stats = filtered_interactions.groupby('recipe_id').agg({
        'rating': ['mean', 'count']
    }).reset_index()
    recipe_stats.columns = ['recipe_id', 'avg_rating', 'rating_count']
    
    v = recipe_stats["rating_count"]
    R = recipe_stats["avg_rating"]
    m = v.quantile(0.90) 
    c = R.mean()
    recipe_stats['w_score'] = ((v * R) + (m * c)) / (v + m)
    popularity_df = recipe_stats.sort_values('w_score', ascending=False)

    recipes = recipes_df

    print(f"Загрузка данных завершена. Пользователь {new_user_id} добавлен.")
    print("Оцененные рецепты:")
    for name in rated_recipe_names:
        print(f" - {name}")

In [4]:
def get_top_popular_recipes(limit: int = 10):
    if popularity_df is None or recipes is None:
        raise RuntimeError("Данные не загружены. Вызовите load_data() сначала.")
    
    top_recipes = popularity_df.head(limit)
    top_recipes_with_names = top_recipes.merge(
        recipes[['id', 'name']], 
        left_on='recipe_id', 
        right_on='id', 
        how='left'
    )
    
    result = []
    for _, row in top_recipes_with_names.iterrows():
        result.append({
            "recipe_id": int(row['recipe_id']),
            "recipe_name": row['name'] if pd.notna(row['name']) else f"Recipe {row['recipe_id']}",
            "average_rating": float(row['avg_rating']) if pd.notna(row['avg_rating']) else None,
            "rating_count": int(row['rating_count']) if pd.notna(row['rating_count']) else None,
            "weighted_score": float(row['w_score']) if pd.notna(row['w_score']) else None
        })
    return result

In [5]:
def get_recommendations_by_keyword(keyword: str, limit: int = 10):
    if recipes is None or popularity_df is None:
        raise RuntimeError("Данные не загружены. Вызовите load_data() сначала.")
    
    keywords = [word.strip().lower() for word in re.split(r'[,\s]+', keyword) if word.strip()]

    matching_recipes = recipes.copy()
    for kw in keywords:
        matching_recipes = matching_recipes[
            matching_recipes['name'].str.lower().str.contains(kw, na=False) |
            matching_recipes['description'].str.lower().str.contains(kw, na=False) |
            matching_recipes['ingredients'].str.lower().str.contains(kw, na=False)
        ]
    
    if matching_recipes.empty:
        return []  # или raise ValueError(f"Рецепты с '{keyword}' не найдены")
    
    matching_with_popularity = matching_recipes.merge(
        popularity_df[['recipe_id', 'w_score', 'avg_rating', 'rating_count']], 
        left_on='id', 
        right_on='recipe_id', 
        how='inner'
    )

    sorted_recipes = matching_with_popularity.sort_values('w_score', ascending=False).head(limit)
    
    result = []
    for _, row in sorted_recipes.iterrows():
        result.append({
            "recipe_id": int(row['id']),
            "recipe_name": row['name'] if pd.notna(row['name']) else f"Recipe {row['id']}",
            "average_rating": float(row['avg_rating']) if pd.notna(row['avg_rating']) else None,
            "rating_count": int(row['rating_count']) if pd.notna(row['rating_count']) else None,
            "weighted_score": float(row['w_score']) if pd.notna(row['w_score']) else None
        })
    return result

In [6]:
def get_content_based_recommendations(recipe_name: str, limit: int = 10):
    if recipe_matrix is None or recipe_map is None or recipe_reverse_map is None or users_pivot is None or recipes is None:
        raise RuntimeError("Данные не загружены. Вызовите load_data() сначала.")

    available_recipe_ids = set(recipe_map.keys())
    available_recipes = recipes[recipes['id'].isin(available_recipe_ids)]
    
    matching_recipes = available_recipes[available_recipes['name'].str.lower().str.contains(recipe_name.lower(), na=False)]

    if matching_recipes.empty:
        return []  # или raise ValueError(f"Рецепт '{recipe_name}' не найден")

    target_recipe = matching_recipes.iloc[0]
    recipe_id = int(target_recipe['id'])
    
    recipe_features = recipe_matrix.T
    model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
    model_knn.fit(recipe_features)
    
    recipe_index = recipe_map[recipe_id]
    
    distances, indices = model_knn.kneighbors(
        recipe_features[recipe_index].toarray(), 
        n_neighbors=min(limit + 1, len(users_pivot.columns))
    )
    
    similar_indices = indices[0][1:]
    distances = distances[0][1:]
    
    result = []
    for idx, dist in zip(similar_indices, distances):
        original_recipe_id = recipe_reverse_map[idx]
        recipe_info = recipes[recipes['id'] == original_recipe_id]
        
        recipe_name_display = f"Recipe {original_recipe_id}"
        if len(recipe_info) > 0 and pd.notna(recipe_info.iloc[0]['name']):
            recipe_name_display = recipe_info.iloc[0]['name']
        
        result.append({
            "recipe_id": int(original_recipe_id),
            "recipe_name": recipe_name_display,
            "similarity_score": float(1 - dist)
        })
    return result

In [28]:
def get_collaborative_recommendations_by_user(user_id: int, limit: int = 10):
    global users_pivot_array, user_map, recipe_reverse_map, recipes, popularity_df

    if users_pivot_array is None or user_map is None or recipe_reverse_map is None or recipes is None:
        raise RuntimeError("Данные не загружены. Вызовите load_data() сначала.")

    if user_id not in user_map:
        top = get_top_popular_recipes(limit)
        return {
            "recommended_recipes": top,
            "user_ratings": []
        }

    user_index = user_map[user_id]  # позиция в массиве (0..N-1)

    # Косинусное сходство по всем пользователям
    user_similarity = cosine_similarity(users_pivot_array)
    similarities = user_similarity[user_index]

    # Находим похожих (кроме себя)
    similar_users = sorted(
        enumerate(similarities),
        key=lambda x: x[1],
        reverse=True
    )[1:limit + 1]

    # Собираем рекомендации
    recommended_recipes = {}
    for i, similarity_score in similar_users:
        similar_user_ratings = users_pivot_array[i]
        for recipe_idx, rating in enumerate(similar_user_ratings):
            if rating > 0 and users_pivot_array[user_index, recipe_idx] == 0:
                original_recipe_id = recipe_reverse_map[recipe_idx]
                if original_recipe_id not in recommended_recipes:
                    recommended_recipes[original_recipe_id] = {'ratings': []}
                recommended_recipes[original_recipe_id]['ratings'].append(rating)

    # Усредняем
    avg_recommendations = []
    for recipe_id, data in recommended_recipes.items():
        avg_rating = np.mean(data['ratings'])
        avg_recommendations.append((recipe_id, avg_rating))

    sorted_recommendations = sorted(avg_recommendations, key=lambda x: x[1], reverse=True)[:limit]

    # Формируем результат
    result = []
    for recipe_id, avg_rating in sorted_recommendations:
        recipe_popularity = popularity_df[popularity_df['recipe_id'] == recipe_id]
        weighted_score = float(recipe_popularity['w_score'].iloc[0]) if not recipe_popularity.empty else None
        rating_count = int(recipe_popularity['rating_count'].iloc[0]) if not recipe_popularity.empty else None
        
        recipe_info = recipes[recipes['id'] == recipe_id]
        recipe_name = f"Recipe {recipe_id}"
        if len(recipe_info) > 0 and pd.notna(recipe_info.iloc[0]['name']):
            recipe_name = recipe_info.iloc[0]['name']

        result.append({
            "recipe_id": int(recipe_id),
            "recipe_name": recipe_name,
            "average_rating": float(avg_rating),
            "weighted_score": weighted_score,
            "rating_count": rating_count
        })

    # Оценки пользователя
    user_ratings = users_pivot_array[user_index]
    rated_recipes = []
    for recipe_idx, rating in enumerate(user_ratings):
        if rating > 0:
            original_recipe_id = recipe_reverse_map[recipe_idx]
            recipe_info = recipes[recipes['id'] == original_recipe_id]
            recipe_name = f"Recipe {original_recipe_id}"
            if len(recipe_info) > 0 and pd.notna(recipe_info.iloc[0]['name']):
                recipe_name = recipe_info.iloc[0]['name']

            rated_recipes.append({
                "recipe_id": int(original_recipe_id),
                "recipe_name": recipe_name,
                "average_rating": float(rating)
            })

    return {
        "recommended_recipes": result,
        "user_ratings": rated_recipes
    }

In [29]:
load_data()

Загрузка данных...
ID рецептов для нового пользователя 666666: [98783, 152116, 135585, 12572, 86858, 8507, 69213, 65266, 32082, 86940, 17083, 34233, 227257, 169916, 53594, 50022, 232055, 273976, 89503, 3579, 130793, 119804, 61800, 30081, 449452, 37413, 79944, 29121, 26110, 31811]
Количество финальных пользователей: 5000 (включая 666666)
Количество финальных рецептов: 3000 (включая [98783, 152116, 135585]...)
Количество строк в filtered_interactions: 139545
Загрузка данных завершена. Пользователь 666666 добавлен.
Оцененные рецепты:
 - chocolate orange fudge
 - deli rotisserie chicken
 - baked buffalo chicken breasts
 - savory crescent chicken squares
 - ground beef goulash
 - mozzarella  tomato and basil salad
 - kittencal s beer battered fish
 - oven fried southern style cinnamon honey chicken
 - chocolate banana cake
 - chicken thighs oven fried
 - yummy banana bread
 - guacamole
 - kittencal s 1 gram low fat banana blueberry muffins
 - whole wheat hamburger and hot dog buns  bread ma

In [30]:
res = get_collaborative_recommendations_by_user(666666, limit=10)

print("Рекомендованные рецепты (10):")
for r in res["recommended_recipes"]:
    print(f"- {r['recipe_name']} (оценка: {r['average_rating']:.2f})")

print(f"\nВсе оценённые рецепты ({len(res['user_ratings'])} шт.):")
for r in res["user_ratings"]:
    print(f"- {r['recipe_name']} → {r['average_rating']}")

Рекомендованные рецепты (10):
- collard greens   it s good for you (оценка: 5.00)
- chinese wontons (оценка: 5.00)
- taco bell taco seasoning clone copycat (оценка: 5.00)
- cornbread with corn casserole (оценка: 5.00)
- grilled eggplant (оценка: 5.00)
- edna s apple crumble  aka  apple crisp (оценка: 5.00)
- cracker barrel s hashbrowns casserole   copycat (оценка: 5.00)
- 1 pan fudge cake (оценка: 5.00)
- ground beef chili (оценка: 5.00)
- weight watchers low fat taco soup (оценка: 5.00)

Все оценённые рецепты (30 шт.):
- the best banana bread → 3.0
- guacamole → 5.0
- beef patties in onion gravy → 5.0
- chocolate banana cake → 3.0
- southern buttermilk biscuits → 5.0
- ground beef gyros → 4.0
- chocolate orange fudge → 3.0
- oven fried southern style cinnamon honey chicken → 4.0
- bacon wrapped chicken  oamc → 5.0
- mozzarella  tomato and basil salad → 3.0
- deli rotisserie chicken → 5.0
- mom s apple crisp → 5.0
- yummy banana bread → 4.0
- vegetarian taco soup → 3.0
- caesar salad →